# GP Sparse Variational Tutorial

Purpose: same deflection problem at larger sample count using inducing points.

- Highlights ELBO trend and inducing-point approximation.


## Learning Roadmap

- Understand inducing-point approximation and ELBO behavior.
- Compare sparse GP uncertainty to exact-GP intuition.
- Diagnose approximation quality using residual and uncertainty plots.


In [ ]:
# Step 1: import dependencies and reproducibility controls
# Configure Python path for local package imports
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'gp' else Path.cwd().resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import math
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)
np.random.seed(42)

from deepuq.models import SparseGaussianProcessRegressor, RBFKernel


In [ ]:
# Step 2: define calibration metrics
def regression_metrics(y_true, mean, var):
    y_true = y_true.reshape(-1)
    mean = mean.reshape(-1)
    var = var.reshape(-1).clamp_min(1e-8)
    rmse = torch.sqrt(torch.mean((mean - y_true) ** 2)).item()
    nll = 0.5 * torch.mean(torch.log(2 * torch.pi * var) + (y_true - mean) ** 2 / var).item()
    std = torch.sqrt(var)
    lower = mean - 1.96 * std
    upper = mean + 1.96 * std
    coverage95 = torch.mean(((y_true >= lower) & (y_true <= upper)).float()).item()
    width95 = torch.mean(upper - lower).item()
    return {
        'rmse': rmse,
        'nll': nll,
        'coverage95': coverage95,
        'interval_width95': width95,
    }


In [ ]:
# Step 3: generate larger-scale synthetic regression data
x_train = torch.linspace(-4, 4, 400).unsqueeze(-1)
true_fn = lambda x: 0.2 * x + torch.sin(1.1 * x) - 0.06 * x**2
y_train = true_fn(x_train) + 0.15 * torch.randn_like(x_train)

x_test = torch.linspace(-6, 6, 300).unsqueeze(-1)
y_test = true_fn(x_test)


In [ ]:
# Step 4: train sparse GP and evaluate metrics
sparse_gp = SparseGaussianProcessRegressor(
    num_inducing=50,
    learning_rate=0.04,
    num_iterations=220,
    kernel=RBFKernel(lengthscale=0.9, outputscale=1.2, jitter=1e-6),
    verbose=False,
)
start = time.perf_counter()
sparse_gp.fit(x_train, y_train)
fit_time = time.perf_counter() - start

uq = sparse_gp.predict_uq(x_test)
metrics = regression_metrics(y_test.squeeze(-1), uq.mean, uq.total_var)
print({'fit_time_sec': round(fit_time, 4), **{k: round(v, 4) for k, v in metrics.items()}})


In [ ]:
# Step 5: inspect ELBO optimization behavior
plt.figure(figsize=(8, 3.2))
plt.plot(sparse_gp.elbo_history)
plt.title('Sparse GP ELBO over iterations')
plt.xlabel('iteration')
plt.ylabel('ELBO')
plt.show()


In [ ]:
# Step 6: visualize sparse GP posterior and inducing points
std = torch.sqrt(uq.total_var)
plt.figure(figsize=(10, 5))
plt.scatter(x_train.numpy(), y_train.numpy(), s=8, alpha=0.30, label='Train')
plt.scatter(
    sparse_gp.inducing_points_[:, 0].detach().numpy(),
    np.full(sparse_gp.inducing_points_.shape[0], y_train.min().item() - 0.3),
    marker='|',
    s=180,
    c='tab:red',
    label='Inducing points',
)
plt.plot(x_test.numpy(), y_test.numpy(), 'k--', lw=1.2, label='True')
plt.plot(x_test.numpy(), uq.mean.numpy(), color='tab:green', lw=2, label='Sparse GP mean')
plt.fill_between(
    x_test.squeeze(-1).numpy(),
    (uq.mean - 1.96 * std).numpy(),
    (uq.mean + 1.96 * std).numpy(),
    alpha=0.2,
    color='tab:green',
    label='95% interval',
)
plt.title('Sparse GP with inducing points')
plt.legend(loc='best')
plt.show()


In [ ]:
# Additional diagnostic: smoothed ELBO and residual profile
# EMA helps reveal ELBO trend when the raw curve is noisy.
elbo = torch.tensor(sparse_gp.elbo_history)
ema = []
alpha = 0.15
for i, v in enumerate(elbo):
    ema.append(v if i == 0 else alpha * v + (1 - alpha) * ema[-1])

residual = uq.mean - y_test.squeeze(-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(elbo.numpy(), alpha=0.35, label='raw ELBO')
axes[0].plot(torch.tensor(ema).numpy(), lw=2, label='EMA ELBO')
axes[0].set_title('Sparse GP ELBO trend')
axes[0].set_xlabel('iteration')
axes[0].set_ylabel('ELBO')
axes[0].legend(loc='best')

axes[1].plot(x_test.numpy(), residual.numpy(), color='tab:red', lw=1.6)
axes[1].axhline(0.0, color='k', ls='--', lw=1)
axes[1].set_title('Residual vs input')
axes[1].set_xlabel('x')
axes[1].set_ylabel('prediction error')
plt.tight_layout()
plt.show()
